### First, we install Librosa's library into Google collab's terminal
```
pip install librosa
```
Then import the library in the code.

In [1]:
import librosa as li
print(li.__version__)

0.11.0


### Downloading the dataset from kaggle
Import kagglehub and pathlib to download the dataset

In [1]:
import kagglehub
from pathlib import Path

genres_original_path = next(Path(kagglehub.dataset_download(
    "andradaolteanu/gtzan-dataset-music-genre-classification")).rglob("genres_original"))

Using Colab cache for faster access to the 'gtzan-dataset-music-genre-classification' dataset.


In [ ]:
import warnings
import numpy as np
import pandas as pd
import soundfile as sf

# Numpy 1.25+ warns when float(array) is used; we convert tempo safely to a scalar.
def to_scalar(value):
    arr = np.asarray(value).reshape(-1)
    return float(arr[0]) if arr.size else float("nan")

# Prefer soundfile (faster/cleaner); fallback to librosa if a file cannot be read there.
def load_audio_safe(file_path, target_sr=22050):
    try:
        y, sr = sf.read(file_path, dtype="float32")
        if y.ndim > 1:
            y = np.mean(y, axis=1)
        if sr != target_sr:
            y = li.resample(y, orig_sr=sr, target_sr=target_sr)
            sr = target_sr
        return y, sr
    except Exception:
        # Hide noisy fallback warnings from audioread/librosa in notebook output.
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", message="PySoundFile failed.*")
            warnings.filterwarnings("ignore", message="librosa.core.audio.__audioread_load.*")
            return li.load(file_path, sr=target_sr, mono=True)

# These are the columns you requested, one row per song.
rows = []
audio_extensions = {".wav", ".mp3", ".au", ".flac", ".ogg", ".m4a"}

for genre_dir in sorted(genres_original_path.iterdir()):
    if not genre_dir.is_dir():
        continue

    # Folder name is the genre label.
    label = genre_dir.name

    for song_path in sorted(genre_dir.iterdir()):
        if song_path.suffix.lower() not in audio_extensions:
            continue

        try:
            y, sr = load_audio_safe(song_path, target_sr=22050)

            # librosa.feature computes time-series descriptors from each audio signal.
            mfcc = li.feature.mfcc(y=y, sr=sr, n_mfcc=20)
            chroma = li.feature.chroma_stft(y=y, sr=sr)
            spectral_centroid = li.feature.spectral_centroid(y=y, sr=sr)
            spectral_rolloff = li.feature.spectral_rolloff(y=y, sr=sr)
            zcr = li.feature.zero_crossing_rate(y)
            tempo, _ = li.beat.beat_track(y=y, sr=sr)

            rows.append({
                "mfcc_std": float(np.std(mfcc)),
                "chroma_mean": float(np.mean(chroma)),
                "spectral_centroid_mean": float(np.mean(spectral_centroid)),
                "spectral_rolloff_mean": float(np.mean(spectral_rolloff)),
                "zero_crossing_rate_mean": float(np.mean(zcr)),
                "tempo": to_scalar(tempo),
                "label": label,
            })
        except Exception as exc:
            print(f"Skipping {song_path.name} ({label}) due to error: {exc}")

features_df = pd.DataFrame(rows, columns=[
    "mfcc_std",
    "chroma_mean",
    "spectral_centroid_mean",
    "spectral_rolloff_mean",
    "zero_crossing_rate_mean",
    "tempo",
    "label",
])

output_csv = Path.cwd() / "gtzan_selected_features.csv"
features_df.to_csv(output_csv, index=False)

print(f"CSV created successfully: {output_csv}")
print(f"Total songs processed: {len(features_df)}")
features_df.head()

Skipping jazz.00054.wav (jazz) due to error: 
CSV created successfully: /content/gtzan_selected_features.csv
Total songs processed: 999


,mfcc_std,chroma_mean,spectral_centroid_mean,spectral_rolloff_mean,zero_crossing_rate_mean,tempo,label
0,42.042160,0.350129,1784.122641,3805.723030,0.083045,123.046875,blues
1,60.053635,0.340849,1530.261767,3550.713616,0.056040,67.999589,blues
2,43.078026,0.363538,1552.832481,3042.410115,0.076291,161.499023,blues
3,59.700958,0.404854,1070.153418,2184.879029,0.033309,63.024009,blues
4,51.301071,0.308526,1835.128513,3579.957471,0.101461,135.999178,blues


In [ ]:
from IPython.display import FileLink, display, Markdown

# Cell 6: download/display helper (run after Cell 5).
csv_path = Path.cwd() / "gtzan_selected_features.csv"

if not csv_path.exists():
    print("CSV file not found yet. Run Cell 5 first to generate it.")
else:
    display(Markdown(f"### Download\nCSV ready at: **{csv_path}**"))

    # Detect if the notebook is running in a Colab runtime.
    in_colab = False
    try:
        import google.colab  # noqa: F401
        in_colab = True
    except Exception:
        in_colab = False

    if in_colab:
        # In Colab, use browser download. /content links often fail inside VS Code UI.
        from google.colab import files
        files.download(str(csv_path))
        print("If your browser blocks popups, allow downloads and run Cell 6 again.")
    else:
        # In local Jupyter/VS Code kernels, show a clickable local file link.
        display(FileLink(str(csv_path)))
        print("Tip: if it opens as text instead of downloading, use right-click > Save as.")

### Download
CSV ready at: **/content/gtzan_selected_features.csv**

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

If your browser blocks popups, allow downloads and run Cell 6 again.
